# ECG v2: CPU → A100 → CPU → A100

Все ветки запускаются явно. Подготовка данных — локально через `pipelines.cpu.run`.
Порядок и датасеты: [FRESH_START](../docs/FRESH_START.md). Старые trainers сохранены как baselines.
Тестовые и external записи не участвуют в подборе моделей. Smoke проверяет исполнение, не качество.


In [ ]:
from pathlib import Path
import os, json, sys, subprocess
if not Path('ecg_project').is_dir() and Path('../ecg_project').is_dir(): os.chdir('..')
assert Path('ecg_project').is_dir(), 'Start in the repository root'
from pipelines.gpu.experiments import require_cache, require_train_sources

RUN_SMOKE = False
RUN_DELINEATION = False
RUN_BERT = False
RUN_FOUNDER = False
RUN_HUBERT_LORA = False
RUN_QWEN = False
RUN_QWEN_CURRICULUM = False  # when RUN_QWEN: replace one distilled run with validation-gated cycles
PSEUDO_RATIOS = [1, 2, 4]
USE_EXTENDED_BEAT_DATASETS = True
USE_EXTENDED_RECORD_DATASETS = True
USE_UNLABELED_SSL = True
USE_QTDB_MANUAL = True
BERT_MODE = 'extended_ssl'  # supervised / mit_masked / extended_ssl
QWEN_EXPERIMENTS = ['B']  # A LUDB; B +QT; C +Training_2 hard; D +extended hard; E +soft KD
QWEN_SIZES = ['1.7B']  # add '4B' for an explicit model-size comparison
HUBERT_VARIANTS = ['lora']  # frozen / lora / qlora
TEACHER = 'artifacts/cluster/delineator_qt.pt'
if any([RUN_DELINEATION,RUN_BERT,RUN_FOUNDER,RUN_HUBERT_LORA,RUN_QWEN]):
    from pipelines.gpu.run import require_a100_memory
    VRAM_GB = require_a100_memory()
else: VRAM_GB = 40
print('GPU branches enabled:', {k:v for k,v in list(globals().items()) if k.startswith('RUN_')})


## CPU 1 — выполнить локально

`python -m pipelines.cpu.run audit --workers 4`

`python -m pipelines.cpu.run prepare-qwen --datasets LUDB QTDB --validation-datasets LUDB QTDB --workers 4`

Для A: та же команда с `--datasets LUDB --output artifacts/qwen_ludb_inputs_v2`.
Для teacher с consistency: отдельный cache `--output artifacts/delineator_inputs_v2 --unlabeled-sources CPSC_EXTRA`.

`python -m pipelines.cpu.run prepare-hubert --workers 4`

`python -m pipelines.cpu.run prepare-founder --workers 4`

Эти команды здесь не выполняются. Перенесите законченные caches и базы Founder/HuBERT на кластер.


In [ ]:
if RUN_DELINEATION:
    from dataclasses import replace
    from ecg_project.training.delineation_v2 import DelineationConfig, train
    teacher_inputs = 'artifacts/delineator_inputs_v2'
    datasets = 'LUDB QTDB' if USE_QTDB_MANUAL else 'LUDB'
    require_cache(teacher_inputs, f'python -m pipelines.cpu.run prepare-qwen --datasets {datasets} --validation-datasets LUDB QTDB --unlabeled-sources CPSC_EXTRA --output {teacher_inputs} --workers 4')
    require_train_sources(teacher_inputs,datasets.split())
    cfg = DelineationConfig(input_root=teacher_inputs, output='artifacts/cluster/delineator', final_checkpoint=TEACHER,
                            batch_size=128 if VRAM_GB>60 else 64, patience=9)
    if RUN_SMOKE: train(replace(cfg,output=cfg.output+'_smoke',final_checkpoint='',epochs=1,max_batches=2,valid_limit=8))
    train(cfg)


## CPU 2 — вернуть новый teacher локально

Checkpoint: `artifacts/cluster/delineator_qt.pt`. После изменения teacher нужен новый output cache.

`python -m pipelines.cpu.run prepare-beats --sources MIT SVDB INCART --checkpoint artifacts/cluster/delineator_qt.pt --workers 4`

`python -m pipelines.cpu.run prepare-unlabeled --sources CPSC_EXTRA PTBXL CPSC CHAPMAN NINGBO --checkpoint artifacts/cluster/delineator_qt.pt --workers 4`

`python -m pipelines.cpu.run prepare-qwen-pseudo --sources CPSC_EXTRA --checkpoint artifacts/cluster/delineator_qt.pt --output artifacts/qwen_pseudo_training2 --workers 4`

`python -m pipelines.cpu.run prepare-qwen-pseudo --sources CPSC_EXTRA PTBXL CPSC CHAPMAN NINGBO --checkpoint artifacts/cluster/delineator_qt.pt --output artifacts/qwen_pseudo_extended --workers 4`

HGB: `prepare-records --checkpoint artifacts/cluster/delineator_qt.pt --workers 4`, затем `train-records` локально.
Отсутствующий явно указанный dataset — ошибка; автоматической подмены нет.


In [ ]:
if RUN_BERT:
    from dataclasses import replace
    from ecg_project.training.cluster_training import ClusterConfig, train_cluster
    assert BERT_MODE in ('supervised','mit_masked','extended_ssl')
    assert BERT_MODE!='extended_ssl' or USE_UNLABELED_SSL, 'Extended SSL requires USE_UNLABELED_SSL'
    beat_root = 'artifacts/beat_features_v2' if USE_EXTENDED_BEAT_DATASETS else 'artifacts/beat_mit_v2'
    sources = 'MIT SVDB INCART' if USE_EXTENDED_BEAT_DATASETS else 'MIT'
    require_cache(beat_root, f'python -m pipelines.cpu.run prepare-beats --sources {sources} --checkpoint {TEACHER} --output {beat_root} --workers 4')
    require_train_sources(beat_root,sources.split())
    ssl_roots = ['artifacts/unlabeled_beats_v2'] if BERT_MODE=='extended_ssl' else []
    for root in ssl_roots:
        require_cache(root, f'python -m pipelines.cpu.run prepare-unlabeled --sources CPSC_EXTRA PTBXL CPSC CHAPMAN NINGBO --checkpoint {TEACHER} --output {root} --workers 4')
    print('BERT mode:', BERT_MODE, 'unlabeled_roots:', ssl_roots)
    cfg = ClusterConfig(data_root=beat_root, output=f'artifacts/cluster/bert_{BERT_MODE}',unlabeled_roots=ssl_roots,
                        pretrain_epochs=0 if BERT_MODE=='supervised' else 30,patience=9)
    if RUN_SMOKE: train_cluster(replace(cfg,output=cfg.output+'_smoke',pretrain_epochs=min(1,cfg.pretrain_epochs),finetune_epochs=1,max_batches=2))
    train_cluster(cfg)


In [ ]:
record_sources = 'PTBXL CPSC CPSC_EXTRA PTB CHAPMAN NINGBO' if USE_EXTENDED_RECORD_DATASETS else 'PTBXL'
if RUN_FOUNDER:
    from ecg_project.training.cluster_founder import train_founder_cluster
    root = 'artifacts/founder_inputs_v2' if USE_EXTENDED_RECORD_DATASETS else 'artifacts/founder_ptb_inputs_v2'
    require_cache(root, f'python -m pipelines.cpu.run prepare-founder --sources {record_sources} --output {root} --workers 4', ('signals.npy','manifest.csv','provenance.json'))
    require_train_sources(root,record_sources.split())
    require_cache('artifacts/ecgfounder','python scripts/download_founder.py --weights',('1_lead_ECGFounder.pth',))
    kwargs = dict(input_root=root,batch_size=128 if VRAM_GB>60 else 64,accumulation=1,patience=9)
    if RUN_SMOKE: train_founder_cluster(output='artifacts/cluster/founder_v2_smoke',epochs=1,max_batches=2,**kwargs)
    train_founder_cluster(output='artifacts/cluster/founder_v2',epochs=100,**kwargs)
if RUN_HUBERT_LORA:
    from dataclasses import replace
    from ecg_project.training.hubert_lora import LoRAConfig, run
    root = 'artifacts/hubert_inputs_v2' if USE_EXTENDED_RECORD_DATASETS else 'artifacts/hubert_ptb_inputs_v2'
    require_cache(root, f'python -m pipelines.cpu.run prepare-hubert --sources {record_sources} --output {root} --workers 4', ('signals.npy','manifest.csv','provenance.json'))
    require_train_sources(root,record_sources.split())
    require_cache('artifacts/hubert_large','python scripts/download_hubert.py --weights',('model.safetensors','config.json','provenance.json'))
    for variant in HUBERT_VARIANTS:
        assert variant in ('frozen','lora','qlora')
        cfg = LoRAConfig(input_root=root,output=f'artifacts/cluster/hubert_{variant}_v2',rank=0 if variant=='frozen' else 16,
                         quantization='nf4' if variant=='qlora' else 'none',batch_size=64 if VRAM_GB>60 else 32,
                         accumulation=2,epochs=100,patience=9,local_minutes=None)
        if RUN_SMOKE: run(replace(cfg,output=cfg.output+'_smoke',epochs=1,max_batches=2,train_limit=64,valid_limit=32))
        run(cfg)


## Qwen supervised / distilled

A и B используют одинаковую LUDB+QTDB validation. C–E добавляют supervision от U-Net; это другой бюджет разметки.
Микробатчи A100: 16 (40 GB), 32 (80 GB), effective manual batch 64. Это начальные профили, не замер максимальной утилизации.
`run.json` сохраняет PyTorch peak memory и throughput. Для 4B при OOM уменьшите batch и увеличьте accumulation в новом run.


In [ ]:
if RUN_QWEN:
    from pipelines.gpu.experiments import qwen_experiment, assert_qwen_inputs, run_qwen
    assert USE_QTDB_MANUAL or QWEN_EXPERIMENTS==['A'], 'B–E require manual QTDB'
    for size in QWEN_SIZES:
        configs = [qwen_experiment(name,size,VRAM_GB) for name in QWEN_EXPERIMENTS]
        for cfg in configs: assert_qwen_inputs(cfg)
        # Only this explicit cluster branch downloads Qwen.
        if not (Path(configs[0].model_root)/'provenance.json').is_file():
            subprocess.run([sys.executable,'scripts/download_qwen.py','--size',size],check=True)
        for cfg in configs:
            if RUN_QWEN_CURRICULUM:
                from dataclasses import replace
                from pipelines.gpu.curriculum import run_curriculum
                from ecg_project.training.delineation_v2 import train
                assert cfg.pseudo_root, 'Curriculum needs C, D or E'
                if RUN_SMOKE: train(replace(cfg,output=cfg.output+'_smoke',epochs=1,max_batches=2,valid_limit=8))
                # To start after an existing E run, set cfg = replace(cfg,warm_start='.../best.pt').
                run_curriculum(cfg,ratios=PSEUDO_RATIOS,max_stale_cycles=2)
            else: run_qwen(cfg,run_smoke=RUN_SMOKE)


## Анализ законченных runs

Таблица содержит validation, не test selection. Качество на test оценивается отдельно после фиксации выбора.
При сравнении U-Net и distilled Qwen обязательно проверить provenance teacher и общий evaluation cohort.


In [ ]:
import pandas as pd
rows=[]
for path in sorted(Path('artifacts/cluster').glob('*/best_metrics.json')):
    metrics=json.loads(path.read_text()); run_path=path.parent/'run.json'
    run=json.loads(run_path.read_text()) if run_path.exists() else {}
    rows.append(dict(run=path.parent.name,regime=metrics.get('training_regime','see config'),
                     dice=metrics.get('valid_macro_wave_dice'),auroc=metrics.get('macro_auroc'),
                     selection_f1=metrics.get('selection_score'),best_epoch=metrics.get('best_epoch'),
                     peak_GiB=run.get('max_memory_allocated',0)/2**30,examples_sec=run.get('examples_per_second'),smoke=run.get('smoke')))
display(pd.DataFrame(rows))


In [ ]:
# Choose a completed delineation run to inspect stored predictions against partial manual labels.
PLOT_RUN = None  # e.g. 'artifacts/cluster/qwen_1.7b_E'
if PLOT_RUN:
    import numpy as np
    import matplotlib.pyplot as plt
    run=Path(PLOT_RUN); cfg=json.loads((run/'config.json').read_text())
    pred=np.load(run/'valid_predictions.npz',allow_pickle=False)
    signals=np.load(Path(cfg['input_root'])/'valid_x.npy',mmap_mode='r')
    sample=0; t=np.arange(signals.shape[-1])/250
    fig,axes=plt.subplots(3,1,figsize=(16,6),sharex=True)
    axes[0].plot(t,signals[sample,0]); axes[0].set_ylabel('ECG')
    axes[1].plot(t,np.where(pred['truth'][sample]>=0,pred['truth'][sample],np.nan)); axes[1].set_ylabel('Manual')
    axes[2].plot(t,pred['prediction'][sample]); axes[2].set_ylabel('Predicted'); axes[2].set_xlabel('Seconds')
    plt.show()
# Export locally or on cluster: python scripts/make_analysis_bundle.py --output analysis_v2.zip


## Record classification validation
Set CLASSIFICATION_RUN to a completed Founder or HuBERT run. No test tuning.

In [ ]:
CLASSIFICATION_RUN = None
if CLASSIFICATION_RUN:
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score
    z = np.load(Path(CLASSIFICATION_RUN)/'valid_predictions.npz', allow_pickle=False)
    truth, probability, classes = z['truth'], z['probability'], z['classes']
    metrics = []
    fig, ax = plt.subplots(figsize=(8,6))
    for i, name in enumerate(classes):
        y, p = truth[:,i], probability[:,i]
        support = int(y.sum())
        row = dict(target=str(name), positive=support, negative=len(y)-support)
        if 0 < support < len(y):
            precision, recall, _ = precision_recall_curve(y,p)
            row.update(auroc=roc_auc_score(y,p), average_precision=average_precision_score(y,p))
            ax.plot(recall,precision,label=str(name))
        metrics.append(row)
    ax.set(xlabel='Recall',ylabel='Precision',title='Validation PR curves')
    ax.legend(); plt.show()
    display(pd.DataFrame(metrics))